# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Join page', 'url': 'https://huggingface.co/join'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
4 days ago
•
321k
•
3.33k
openai/privacy-filter
Updated
9 days ago
•
92.6k
•
1.15k
deepseek-ai/DeepSeek-V4-Flash
Updated
4 days ago
•
281k
•
895
XiaomiMiMo/MiMo-V2.5-Pro
Updated
3 days ago
•
7.94k
•
326
Qwen/Qwen3.6-27B
Updated
7 days ago
•
907k
•
1.04k
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
266
ML Intern
🤖
266
Chat with an AI intern for machine learning help
Running
on
Zero
MCP
2.49k
Wan2.2 14B Preview
🐌
2.49k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured

In [20]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI and machine learning community platform committed to building the future of artificial intelligence through collaboration and open innovation. It serves as the central hub where machine learning enthusiasts, engineers, scientists, and end users can come together to create, share, and advance models, datasets, and AI applications.

---

## Our Mission

Hugging Face’s mission is to empower the next generation of machine learning professionals and researchers by providing a collaborative platform that fosters open and ethical AI development.

---

## What We Offer

- **Vast Model Repository:** Access over 2 million pre-trained models spanning modalities including text, image, video, audio, and 3D.
- **Extensive Datasets:** Explore more than 500,000 datasets that fuel diverse machine learning projects.
- **Interactive Spaces:** Deploy and run machine learning applications quickly while collaborating with the community.
- **Open-Source Tools:** Benefit from the Hugging Face open-source stack which accelerates ML development and innovation.
- **Collaboration Platform:** Host and collaborate on unlimited public models, datasets, and applications worldwide.

---

## Why Choose Hugging Face?

- **Community-Driven:** With a fast-growing global community, Hugging Face offers an ecosystem for sharing knowledge and resources.
- **Cutting-Edge Innovations:** Stay ahead with access to trending and state-of-the-art machine learning models and applications.
- **Multi-Modal Support:** Work seamlessly with various data types including text, images, video, audio, and 3D.
- **Portfolio Building:** Share your work, contribute to open source, and build a professional machine learning profile that can boost your career.

---

## Company Culture

At Hugging Face, we believe in openness, inclusivity, and collaboration. Our community-centric culture promotes ethical AI practices and supports learning and innovation for all skill levels. We encourage curiosity, continuous improvement, and teamwork to build trustworthy AI that benefits everyone.

---

## Our Customers & Community

Hugging Face serves a diverse user base including:

- Machine learning engineers and researchers
- Data scientists
- AI startups and enterprises
- Academic institutions
- Open-source contributors and hobbyists

Our platform supports businesses and individuals by simplifying the development and deployment of AI models, fostering innovation, and accelerating time-to-market.

---

## Careers at Hugging Face

Join us to be part of a vibrant, innovative, and inclusive AI community. We are looking for talented and passionate individuals who want to make an impact in the machine learning world. Opportunities span across engineering, data science, research, community management, and more.

- **Why Work with Us?**
  - Work on the cutting edge of AI
  - Collaborate with a global community of experts
  - Contribute to open-source projects that drive real-world AI adoption
  - Grow your career in a supportive, mission-driven environment

Visit our careers page to explore current openings and learn how you can join the team.

---

## Contact & Explore More

- Explore AI apps and models: [Hugging Face Platform](https://huggingface.co)
- Join the community, share your work, and accelerate your machine learning journey.
- For brands and enterprises, learn about our enterprise solutions tailored for your AI needs.

---

**Hugging Face**  
*The AI community building the future.*  
Connect, collaborate, and create with Hugging Face.  

---

_Colors and Logo Guidelines_  
- Brand colors: Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)  
- Logo assets available in .svg, .png, and .ai formats  

---

*Empowering open and ethical AI innovation for everyone.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face: The AI Community That's Hugging the Future (Literally!)

---

## Who We Are  
Welcome to **Hugging Face**, where artificial intelligence isn’t just a buzzword — it’s a community movement. If AI was a party, consider us the ultimate host, bringing together over 2 million models, countless datasets, and thousands of enthusiastic AI applications (all comfy in one place). Our mission? Building the future by collaborating, innovating, and sometimes joking about machine learning — because who says algorithms can't have a sense of humor?

---

## What We Offer  

### The Playground for Machine Learning Enthusiasts  
- **Models Galore:** Browse from over 2 million models. Need a video created from an image? A snazzy chatbot intern? Or maybe a high-fidelity 3D generator? It’s all here — updated daily by the community and open source wizards.  
- **Datasets Central:** Half a million datasets covering everything under the AI sun, from reasoning tasks to professional health benchmarks.  
- **Spaces:** Our magical Zones where your AI apps run, whether you're on a CPU Upgrade or ZeroGPU, we’ve got your back with blazing fast deployments!  
- **Buckets:** Store your precious AI treasures with us (think of it as an AI cloud attic).  

---

## Enterprise Love  

*For the AI pros and power-users,*  
Hugging Face Enterprise scales your organization’s AI game with:  
- **Team & Enterprise Plans starting at $20/user/month**  
- **Enterprise-grade security** (because your AI deserves Fort Knox)  
- **Single Sign-On, Audit Logs & Token Management** for seamless and secure teamwork  
- **Advanced Compute Options** like ZeroGPU Quota Boost (5x more! Yes, please)  
- **Priority Support:** Our AI whisperers are on call when you need a human touch  
- **Billing and Analytics Dashboards** to keep your budgets and usage laser-focused

---

## Pricing  

From solo ML rockstars (just $9/month for Pro features) to growing teams and enterprise giants, we've got plans tailored to your AI appetite:  
- **Pro Account:** Amp up your AI solo journey with 10× private storage, inference credit boosts, and a cool PRO badge to wear proudly.  
- **Team Plan:** Instant setup, collaboration tools, and enough compute juice to power AI dreams.

---

## Culture: More than Just Code  

At Hugging Face, we’re not just about silicon and algorithms — we're an AI *family*. We celebrate open source collaboration, value intellectual curiosity, and believe humor and humanity belong in AI conversations. Our community is diverse, welcoming, and passionate, whether you’re a seasoned ML researcher or a curious intern chatting with AI bots.  

- Love sharing your work? Build your **ML portfolio** and showcase your skills.  
- Need help? Our **ML Intern chatbot** is there 24/7!  
- Want to collaborate? Host unlimited public models, datasets, and applications — teamwork makes the dream work!  

---

## Careers & Recruiting  

Passionate about shaping the machine-learning landscape?  
Hugging Face is the place where driven minds come together to push AI boundaries—with an inclusive, innovative, and flexible work environment. We hire engineers, researchers, community managers, and visionaries who want to make AI accessible and delightful for everyone.  

*Join us if you:*  
- Love open source and open minds  
- Enjoy tackling real-world AI challenges  
- Prefer working where creativity meets top-tier AI infrastructure

---

## Why Hugging Face?  

Because we’re not just *building* AI, we’re building the **future** — with kindness, collaboration, and a dash of wit. From solo coders to Fortune 500 companies, everyone gets to join the party, share their models, train together in the cloud, and watch their AI dreams hug reality.

So why just build AI when you can *Hug* AI?

---

**Dive in at**: [huggingface.co](https://huggingface.co)  
Your future AI best friend is waiting.

---

*P.S. We promise the models don't bite. But the community might hug you.* 🤗

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>